# Scripted + silver spontaneous, second attempt

The first attempt diverged: training loss climbed 0.13 to 1.03 and dev error
rose after the first epoch. Three changes:

1. Learning rate 1e-5 rather than 1e-4. 1e-4 is the rate for fine-tuning the
   pretrained encoder; applied to an already-converged checkpoint it destroys
   it, and a decaying schedule does not recover.
2. No warmup. Warmup protects a fresh head; this head is trained.
3. Duration cap raised from 20 s to 30 s and the silver dev split folded in,
   taking spontaneous training audio from 12.21 h to 25.26 h. The organisers'
   silver test split is held out as the spontaneous dev set: 244 clips, against
   221 spontaneous clips in the blind set.

Decoding is the manual CTC collapse.


In [ ]:
import os, json, csv, math, time, random, sys
from pathlib import Path

WORK  = Path('/kaggle/working')
CKPT  = WORK/'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
STATE = WORK/'state.json'

CFG = dict(
    model   = 'facebook/w2v-bert-2.0',
    ft_lr   = 1e-5,  ft_epochs = 4,   # 1e-4 diverged on a converged checkpoint
    batch = 4, accum = 8, warmup = 0.0, weight_decay = 0.01,
    max_sec = 30.0, seed = 1,
)

HF_TOKEN = HF_USER = HF_REPO = None
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi as _HfApi
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    HF_USER  = _HfApi(token=HF_TOKEN).whoami()['name']
    HF_REPO  = f'{HF_USER}/qxp-vs2'
    print('checkpointing to', HF_REPO)
except Exception as e:
    print(f'NO REMOTE CHECKPOINTS ({type(e).__name__}: {e})')
    print('a crash will lose this run. Attach HF_TOKEN and rerun to be safe.')

def state():
    return json.loads(STATE.read_text()) if STATE.exists() else {}
def mark(k, v=True):
    s = state(); s[k] = v; STATE.write_text(json.dumps(s, indent=1))

In [ ]:
!pip -q install 'transformers>=4.44' 'torchaudio' 'jiwer' 2>&1 | tail -1
import torch, transformers
print(transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Token comes from Kaggle Secrets. Never paste it into a cell: cells are saved
# into version history, which is how the token in notebookb2d7b39cba leaked.
from huggingface_hub import HfApi, snapshot_download

HF = HfApi(token=HF_TOKEN) if HF_REPO else None
if HF:
    HF.create_repo(HF_REPO, repo_type='model', private=True, exist_ok=True)
    print('checkpointing to', HF_REPO)
else:
    print('HF_USER is None -> no remote checkpoints. A crash loses the run.')

def push(d, tag, ep):
    """Weights only. Optimizer state stays local: 4.8 GB per epoch of Adam
    moments costs more upload time than the epoch it would save."""
    if not HF:
        return
    for attempt in range(3):
        try:
            HF.upload_folder(folder_path=str(d), repo_id=HF_REPO, path_in_repo=tag,
                             allow_patterns=['*.safetensors', '*.json'],
                             commit_message=f'{tag} epoch {ep}')
            print(f'  pushed {tag} epoch {ep}', flush=True)
            sync_small()
            return
        except Exception as e:
            print(f'  push failed ({attempt+1}/3): {e}', flush=True)
            time.sleep(20)
    print('  push gave up; local copy still on disk', flush=True)

def sync_small():
    """state.json is tiny, and without it a recovered checkpoint does not know
    which epoch it stopped at."""
    if not HF:
        return
    for f in (STATE,):
        if f.exists():
            try:
                HF.upload_file(path_or_fileobj=str(f), path_in_repo=f.name,
                               repo_id=HF_REPO, commit_message=f'sync {f.name}')
            except Exception as e:
                print(f'  sync {f.name} failed: {e}', flush=True)


def restore_small():
    if not HF:
        return
    for name in ('state.json',):
        if (WORK/name).exists():
            continue
        try:
            snapshot_download(HF_REPO, allow_patterns=[name],
                              local_dir=str(WORK), token=HF.token)
            if (WORK/name).exists():
                print(f'recovered {name} from HF')
        except Exception:
            pass


def fetch(tag):
    """Local checkpoint if present, else pull it back from HF. Returns a path
    or None. This is what makes a fresh session resumable."""
    d = CKPT/tag
    if (d/'config.json').exists():
        return d
    if HF:
        try:
            snapshot_download(HF_REPO, allow_patterns=[f'{tag}/*'],
                              local_dir=str(CKPT), token=HF.token)
            if (d/'config.json').exists():
                print(f'recovered {tag} from HF')
                return d
        except Exception as e:
            print(f'no {tag} on HF ({e})')
    return None

restore_small()
print('state:', state())

In [ ]:
# Resolve audio by basename: Kaggle shards uploads across directories. The OOD
# clips are wav, everything else mp3.
IDX = WORK/'audio_index.json'
if IDX.exists():
    AUDIO = json.loads(IDX.read_text())
else:
    AUDIO = {}
    for ext in ('*.mp3', '*.wav'):
        AUDIO.update({q.name: str(q) for q in Path('/kaggle/input').rglob(ext)})
    IDX.write_text(json.dumps(AUDIO))
print(f'indexed {len(AUDIO)} audio files')

## Data

In [ ]:
SCORE_STRIP = set('?!¿¡.,')
def normalise(t):
    return ''.join(c for c in str(t).lower() if c not in SCORE_STRIP).strip()

def read_tsv(p):
    with open(p, encoding='utf-8') as fh:
        return list(csv.DictReader(fh, delimiter='\t', quoting=csv.QUOTE_NONE))

EXPECT = dict(train=18168, dev=2284, test=2275)

def find_splits():
    """Match on row counts: Kaggle nests mounts, and Common Voice ships its own
    train/dev/test under the same filenames."""
    seen = []
    for cand in sorted({q.parent for q in Path('/kaggle/input').rglob('train.tsv')}):
        if not all((cand/f'{k}.tsv').exists() for k in EXPECT):
            continue
        got = {k: sum(1 for _ in open(cand/f'{k}.tsv', encoding='utf-8')) - 1 for k in EXPECT}
        seen.append((str(cand), got))
        if got == EXPECT:
            return cand
    raise AssertionError(f'sentence-disjoint splits not found. candidates: {seen}')

def find_one(name):
    hits = sorted(Path('/kaggle/input').rglob(name))
    assert hits, f'{name} not found -- attach qxp-silver-and-ood'
    return hits[0]

SPLIT_DIR = find_splits()
scripted = {k: [{'path': r['path'], 'sentence': normalise(r['sentence'])}
                for r in read_tsv(SPLIT_DIR/f'{k}.tsv')] for k in EXPECT}

# Silver spontaneous and the OOD set are already filtered: clips over max_sec are
# excluded rather than truncated (truncating audio while keeping the whole
# transcript teaches the model to drop its tail), and clips whose text needs
# characters outside the 36-symbol head are dropped.
silver = {k: [{'path': r['path'], 'sentence': r['sentence']}
              for r in read_tsv(find_one(f'silver2_{k}.tsv'))] for k in ('train','dev')}
ood = [{'path': r['path'], 'sentence': r['sentence']} for r in read_tsv(find_one('ood_dev.tsv'))]

train = scripted['train'] + silver['train']
DEV = {'scripted': scripted['dev'], 'spontaneous': silver['dev'], 'ood': ood}

print(f"train      {len(train):6d}  ({len(scripted['train'])} scripted + {len(silver['train'])} silver)")
for k, v in DEV.items():
    print(f'dev {k:11s}{len(v):6d}')
assert len(train) == 18168 + 6442, len(train)

CKPT_VOCAB = find_one('vocab.json')
vocab = json.loads(CKPT_VOCAB.read_text(encoding='utf-8'))
(WORK/'vocab.json').write_text(json.dumps(vocab, ensure_ascii=False))
print('vocab', len(vocab), 'symbols from', CKPT_VOCAB)

missing = [r['sentence'] for r in train if set(r['sentence']) - {' '} - set(vocab)]
assert not missing, f'{len(missing)} training targets need symbols outside the head'

In [ ]:
import torchaudio, numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2BertForCTC, SeamlessM4TFeatureExtractor, Wav2Vec2CTCTokenizer

tok  = Wav2Vec2CTCTokenizer(str(WORK/'vocab.json'), unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|')
feat = SeamlessM4TFeatureExtractor.from_pretrained(CFG['model'])
BLANK = tok.pad_token_id

def load_audio(path, max_sec):
    w, sr = torchaudio.load(str(path))
    if w.shape[0] > 1: w = w.mean(0, keepdim=True)
    if sr != 16000: w = torchaudio.functional.resample(w, sr, 16000)
    w = w.squeeze(0)[: int(max_sec*16000)]
    return w.numpy()

class Utt(Dataset):
    def __init__(self, rows, augment=False):
        self.rows, self.augment = rows, augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        y = load_audio(AUDIO[Path(r['path']).name], CFG['max_sec'])
        if self.augment:
            f = random.choice([0.9, 1.0, 1.1])
            if f != 1.0:
                n = int(len(y)/f)
                y = np.interp(np.linspace(0, len(y)-1, n), np.arange(len(y)), y).astype('float32')
        x = feat(y, sampling_rate=16000, return_tensors='pt').input_features[0]
        ids = tok(normalise(r['sentence'])).input_ids
        return x, torch.tensor(ids)

def collate(batch):
    xs, ys = zip(*batch)
    T = max(x.shape[0] for x in xs)
    X = torch.zeros(len(xs), T, xs[0].shape[1])
    mask = torch.zeros(len(xs), T, dtype=torch.long)
    for i, x in enumerate(xs):
        X[i, :x.shape[0]] = x; mask[i, :x.shape[0]] = 1
    L = max(len(y) for y in ys)
    Y = torch.full((len(ys), L), -100, dtype=torch.long)
    for i, y in enumerate(ys): Y[i, :len(y)] = y
    return X, mask, Y

## Model and training loop

In [ ]:
def build_model():
    m = Wav2Vec2BertForCTC.from_pretrained(
        CFG['model'], vocab_size=len(vocab), ignore_mismatched_sizes=True,
        ctc_loss_reduction='mean', pad_token_id=BLANK,
        add_adapter=True,
        attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0,
        layerdrop=0.0, mask_time_prob=0.0)
    m.gradient_checkpointing_enable()
    return m.cuda()


def load_stage(tag, fallback):
    """Resume a stage from wherever it survived, else start it fresh."""
    d = fetch(tag)
    if d is None:
        return fallback(), 0
    m = Wav2Vec2BertForCTC.from_pretrained(d).cuda()
    m.gradient_checkpointing_enable()
    ep = state().get(f'{tag}_epoch', -1) + 1
    print(f'{tag}: resuming at epoch {ep}')
    return m, ep


def run_epochs(model, rows, lr, epochs, tag, augment=True, start=0, select=True):
    """Checkpoints after every epoch and pushes the weights off-box. Worst case
    a crash costs the epoch in flight, not the stage.

    Selects the epoch with the lowest mean dev WER across the three domains,
    which is what the leaderboard ranks on."""
    dl = DataLoader(Utt(rows, augment), batch_size=CFG['batch'], shuffle=True,
                    collate_fn=collate, num_workers=2, drop_last=True)
    steps = (len(dl)//CFG['accum'])*epochs
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=lr, weight_decay=CFG['weight_decay'])
    sch = transformers.get_linear_schedule_with_warmup(opt, int(CFG['warmup']*steps), steps)
    scaler = torch.amp.GradScaler('cuda')

    d = CKPT/tag; d.mkdir(parents=True, exist_ok=True)
    trainer = d/'trainer.pt'
    if start and trainer.exists():
        st = torch.load(trainer, map_location='cpu')
        opt.load_state_dict(st['opt']); sch.load_state_dict(st['sch'])
        scaler.load_state_dict(st['scaler'])
        print(f'{tag}: optimizer state restored')
    elif start:
        # Weights came back from HF but the moments did not. Adam rebuilds them
        # within a few dozen steps; the alternative is redoing the whole stage.
        print(f'{tag}: no optimizer state, continuing with fresh moments')

    model.train(); step = start*(len(dl)//CFG['accum']); t0 = time.time()
    for ep in range(start, epochs):
        for i, (X, mask, Y) in enumerate(dl):
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = model(input_features=X.cuda(), attention_mask=mask.cuda(), labels=Y.cuda()).loss
            scaler.scale(loss/CFG['accum']).backward()
            if (i+1) % CFG['accum'] == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sch.step(); step += 1
                if step % 50 == 0:
                    print(f'{tag} ep{ep} step {step}/{steps} loss {loss.item():.3f} {time.time()-t0:.0f}s', flush=True)
        if not select:
            model.save_pretrained(d)
        else:
            m = evaluate_all(model, f'{tag} ep{ep}')
            best = state().get(f'{tag}_best')
            if best is None or m['mean']['wer'] < best['mean']['wer']:
                model.save_pretrained(d)
                mark(f'{tag}_best', dict(epoch=ep, **m))
                print(f"  new best at epoch {ep}: mean WER {m['mean']['wer']:.2f}", flush=True)
            else:
                print(f"  epoch {ep} mean WER {m['mean']['wer']:.2f} >= best "
                      f"{best['mean']['wer']:.2f} (epoch {best['epoch']}), not saved", flush=True)
        torch.save({'opt': opt.state_dict(), 'sch': sch.state_dict(),
                    'scaler': scaler.state_dict(), 'epoch': ep}, trainer)
        mark(f'{tag}_epoch', ep)
        push(d, tag, ep)
        model.train()
    if select and (d/'config.json').exists():
        model = Wav2Vec2BertForCTC.from_pretrained(d).cuda()
        model.gradient_checkpointing_enable()
    return model

def _warm(path):
    m = Wav2Vec2BertForCTC.from_pretrained(path)
    m.gradient_checkpointing_enable()
    return m.cuda()

In [ ]:
import jiwer, re

INV = {v: k for k, v in vocab.items()}
BLANK_ID = vocab['[PAD]']

def collapse(seq):
    """train_qxp.py's CTC collapse. tok.decode(skip_special_tokens=True) scores
    4.41 / 28.37 on a checkpoint this returns 3.07 / 19.92 for."""
    prev, chars = None, []
    for t in seq:
        if t != prev and t != BLANK_ID:
            chars.append(INV.get(t, ''))
        prev = t
    return re.sub(r'\s+', ' ', ''.join(chars).replace('|', ' ')).strip()

@torch.no_grad()
def evaluate(model, rows, tag):
    model.eval(); hyps = []
    dl = DataLoader(Utt(rows), batch_size=CFG['batch'], collate_fn=collate, num_workers=2)
    for X, mask, Y in dl:
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(input_features=X.cuda(), attention_mask=mask.cuda()).logits
        out_len = model._get_feat_extract_output_lengths(mask.sum(-1)).tolist()
        for k, ids in enumerate(logits.argmax(-1).cpu().tolist()):
            hyps.append(collapse(ids[: out_len[k]]))
    refs = [r['sentence'] for r in rows]
    cer = jiwer.cer(refs, hyps)*100; wer = jiwer.wer(refs, hyps)*100
    print(f'  {tag:12s} CER {cer:6.2f}  WER {wer:6.2f}  (n={len(rows)})', flush=True)
    return cer, wer, hyps

def evaluate_all(model, tag=''):
    """The blind set is scripted + spontaneous only, so the mean is over those
    two. OOD is still reported: it is the one proxy we have for unseen
    recording conditions."""
    out = {}
    print(f'{tag} dev:', flush=True)
    for name, rows in DEV.items():
        c, w, _ = evaluate(model, rows, name)
        out[name] = dict(cer=c, wer=w)
    out['mean'] = dict(cer=(out['scripted']['cer'] + out['spontaneous']['cer'])/2,
                       wer=(out['scripted']['wer'] + out['spontaneous']['wer'])/2)
    print(f"  {'MEAN (s+sp)':12s} CER {out['mean']['cer']:6.2f}  WER {out['mean']['wer']:6.2f}", flush=True)
    return out

## Fine-tune on scripted + silver spontaneous

In [ ]:
PRETRAINED = next((q.parent for q in Path('/kaggle/input').rglob('model.safetensors')), None)
assert PRETRAINED is not None, 'attach qxp-w2vbert-seed1'
print('warm start from', PRETRAINED)

if not state().get('vs2'):
    torch.manual_seed(CFG['seed']); random.seed(CFG['seed'])
    model, start = load_stage('vs2', lambda: _warm(PRETRAINED))
    model = run_epochs(model, train, CFG['ft_lr'], CFG['ft_epochs'], 'vs2', start=start)
    mark('vs2', state().get('vs2_best'))
    del model; torch.cuda.empty_cache()
print(json.dumps(state().get('vs2'), indent=1))

## Result

In [ ]:
BASE = dict(scripted=dict(cer=3.07, wer=19.92))   # seed-1, manual decode
vs = state().get('vs2') or {}
print(f"{'domain':14s} {'CER':>7s} {'WER':>7s}")
for k in ('scripted', 'spontaneous', 'ood', 'mean'):
    if k in vs:
        print(f"{k:14s} {vs[k]['cer']:7.2f} {vs[k]['wer']:7.2f}")
if 'scripted' in vs:
    print(f"\nscripted WER vs seed-1: {vs['scripted']['wer'] - BASE['scripted']['wer']:+.2f}")
    print('The first V+S run reached mean 39.86 on scripted+spontaneous with the')
    print('tokenizer decode, from a diverging checkpoint. Any improvement here')
    print('must be read against a fresh seed-1 measurement, not that number.')